# Module 9: Designing a Reproducible Data Integration Pipeline

**Unit E · Week 9 (part 1)** · Track 1 — Data Integration, Standards, Metadata & Quality

Assembles ingest, standardize, validate, document, and publish into one reproducible, re-runnable pipeline — and this is the one that matters most for this course: its published output is data/processed/track2_dataset.csv, the exact file Track 2 reads.

## Learning objectives

- Assemble ingest, standardize, validate, document, and publish steps into one reproducible, re-runnable pipeline instead of manual point-and-click steps.
- Apply basic version control to pipeline outputs via dated, versioned filenames and a run log.


## Setup

This notebook reads the raw practice files in `../../../data/raw/`, built by
`data/make_track1_sources.py` — three partner-style exports shaped like a
real regional hub's source-system landscape (an ERP/financial export, a
survey-platform export, and an HDX-style pull), plus the P-code gazetteer
used to reconcile them. All four are **synthetic**; see `data/README.md`.
Run `python3 data/make_track1_sources.py` once from the repo root before
working through this notebook if those files aren't there yet.

## Lesson content

See the Python notebook for the full lesson content — identical in both languages.

In [ ]:
library(tidyverse)

run_log <- list()

run_step <- function(name, fn) {
  result <- tryCatch({
    r <- fn()
    run_log[[name]] <<- "OK"
    r
  }, error = function(e) {
    run_log[[name]] <<- paste("FAILED:", conditionMessage(e))
    stop(e)
  })
  result
}

ALIASES <- c("Nyarugenge Dist." = "Nyarugenge", "Rwamagana " = "Rwamagana")

ingest <- function() {
  list(
    a = read_csv("../../../data/raw/partner_a_finance_export.csv", show_col_types = FALSE),
    b = read_csv("../../../data/raw/partner_b_survey_export.csv", show_col_types = FALSE),
    c = read_csv("../../../data/raw/partner_c_hdx_pull.csv", show_col_types = FALSE),
    gaz = read_csv("../../../data/raw/cod_ab_gazetteer.csv", show_col_types = FALSE)
  )
}

standardize <- function(sources) {
  a <- sources$a %>% mutate(reg = recode(str_trim(reg), !!!ALIASES),
                             date = as.Date(date, format = "%d/%m/%Y")) %>%
    left_join(sources$gaz, by = c("reg" = "district_name"))
  b <- sources$b %>% mutate(region_name = recode(str_trim(region_name), !!!ALIASES),
                             date = as.Date(collection_date, format = "%d-%b-%y")) %>%
    left_join(sources$gaz, by = c("region_name" = "district_name"))
  c <- sources$c %>% mutate(date = as.Date(date))

  merged <- c %>%
    left_join(a %>% select(district_pcode, date, rev), by = c("district_pcode", "date")) %>%
    left_join(b %>% select(district_pcode, date, feature_1, feature_2), by = c("district_pcode", "date")) %>%
    left_join(sources$gaz %>% select(district_pcode, district_name, province), by = "district_pcode")

  merged %>%
    transmute(date = format(date, "%Y-%m-%d"), province, district = district_name, district_pcode,
              indicator, value = round(rev, 2), feature_1 = round(feature_1, 2),
              feature_2 = round(feature_2, 2), outcome) %>%
    arrange(district, date)
}

sources <- run_step("ingest", ingest)
published <- run_step("standardize", ~ standardize(sources))
run_step("validate", ~ stopifnot(all(published$value >= 0)))
run_step("document", ~ write_csv(published, "../../../data/processed/track2_dataset_DICTIONARY_r.csv"))

write_csv(published, "../../../data/processed/track2_dataset.csv")
timestamp <- Sys.Date()
write_csv(published, paste0("../../../data/processed/published_regional_data_", timestamp, ".csv"))
print(enframe(unlist(run_log), "step", "status"))
cat("\nPublished", nrow(published), "rows to data/processed/track2_dataset.csv\n")

## Your turn

Run this pipeline end-to-end on a fresh clone (after `python3 data/make_track1_sources.py`), confirm the run log shows every stage OK, and diff the published `track2_dataset.csv` against the copy produced by `data/make_sample_data.py` — they should match exactly.

**Formative assessment.** Pipeline runs successfully end-to-end on a held-out test dataset (graded live); run log correctly captures pass/fail of each stage.